# 第 1–2 天：看懂声呐数据

目标：在写训练代码之前，用数字回答「这批数据长什么样」。每一节最后都有一个问题，把答案记进你的数据笔记。

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sonar.plotting import setup_cjk_font
plt.rcParams["figure.dpi"] = 110
setup_cjk_font()   # 服务器上没有中文字体时会提示安装方法

# 下载了真实数据就改成 data/raw/UATD_Training；没下载就先用合成数据（python scripts/make_synthetic.py）
DATA_ROOT = REPO / "data" / "raw" / "UATD_Training"
if not DATA_ROOT.exists():
    DATA_ROOT = REPO / "data" / "synthetic" / "UATD_Training"
print("使用数据：", DATA_ROOT)

## 1. 读数据，看 XML 里有哪些声呐参数

In [ ]:
from sonar.uatd import find_pairs, collect_classes
anns = find_pairs(DATA_ROOT)
classes = collect_classes(anns)
print(f"{len(anns)} 张图，{len(classes)} 个类别：{classes}")
print("\n第一张图的附加字段：")
for k, v in anns[0].meta.items():
    print(f"  {k}: {v}")

> 问题：哪个字段是量程（最大探测距离）？用「量程 ÷ 图像高度」算出每个像素代表多少米。

## 2. 类别、框大小、目标距离的分布

In [ ]:
from sonar.stats import boxes_table
bt = boxes_table(anns, near_at_top=True)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
bt["cls"].value_counts().plot.barh(ax=ax[0], title="各类别框数")
bt["sqrt_area"].plot.hist(bins=40, ax=ax[1], title="框大小（面积开方，像素）")
bt["range_frac"].plot.hist(bins=20, ax=ax[2], title="目标在距离方向的位置（0=近，1=远）")
plt.tight_layout(); plt.show()
print(bt.groupby("range_bin", observed=False)["sqrt_area"].describe().round(1))

> 问题：小于 16 像素的框占多少？远处目标是不是更小？哪个类别样本最少？

## 3. 为什么原图「几乎全黑」：看像素强度分布

In [ ]:
from sonar.io import read_gray, to_uint8
img = read_gray(anns[0].image_path)
print("dtype:", img.dtype, " shape:", img.shape)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].hist(img.ravel(), bins=256, log=True); ax[0].set_title("像素强度直方图（纵轴对数）")
pct = np.percentile(img, [50, 90, 99, 99.9])
ax[1].bar(["P50", "P90", "P99", "P99.9"], pct); ax[1].set_title("分位数")
plt.tight_layout(); plt.show()
print("P50/P90/P99/P99.9 =", pct, " 最大值 =", img.max())

> 问题：中位数是最大值的百分之几？如果只做线性缩放，一半的像素会落在 0–255 的哪一段？

## 4. 原图、增强图、扇形图对比

In [ ]:
from sonar.preprocess import sonar_preprocess
from sonar.geometry import polar_to_fan
from sonar.viz import draw_boxes
a = next(x for x in anns if x.boxes)
raw = read_gray(a.image_path)
enh = sonar_preprocess(raw, "log_clahe")
fan = polar_to_fan(enh, fov_deg=120)   # 开角以你的数据为准
fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))
ax[0].imshow(draw_boxes(a, classes=classes)); ax[0].set_title("原图（线性缩放）")
ax[1].imshow(draw_boxes(a, img=enh, classes=classes)); ax[1].set_title("对数 + CLAHE")
ax[2].imshow(fan, cmap="gray"); ax[2].set_title("展开成扇形（声呐在底部中央）")
for x in ax: x.axis("off")
plt.tight_layout(); plt.show()

> 问题：原图里你能看见几个目标？增强图里呢？扇形图里远处的目标为什么变宽了？

## 5. 找声影：穿过目标中心的单列强度剖面

亮峰后面紧跟一段暗谷，暗谷所在的一侧就是「更远」的方向。

In [ ]:
b = max(a.boxes, key=lambda b: b.w * b.h)
col = int(b.cx)
profile = sonar_preprocess(raw, "log")[:, col].astype(float)
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(profile, lw=0.8)
ax.axvspan(b.y1, b.y2, color="tab:orange", alpha=0.3, label=f"目标 {b.cls}")
ax.axvspan(b.y2, min(len(profile) - 1, b.y2 + 3 * b.h), color="tab:blue", alpha=0.15, label="预期的声影区")
ax.set_xlabel("行号"); ax.set_ylabel("对数强度"); ax.legend(); plt.tight_layout(); plt.show()
before = profile[max(0, int(b.y1 - 2 * b.h)):int(b.y1)].mean()
after = profile[int(b.y2):int(b.y2 + 2 * b.h)].mean()
print(f"目标前方平均强度 {before:.1f}，目标后方平均强度 {after:.1f}")
print("结论：", "行号越大越远（near_at_top=True）" if after < before else "方向可能相反，检查一下 near_at_top")

> 问题：换几个目标再看，结论稳定吗？

## 6. 连续帧有多像：为第 3 天的划分做准备

In [ ]:
from sonar.split import _thumb
t = np.stack([_thumb(x) for x in anns[:60]])
sim = t @ t.T
plt.figure(figsize=(4.5, 4)); plt.imshow(sim, vmin=0, vmax=1); plt.colorbar()
plt.title("前 60 帧两两相似度"); plt.show()
print("相邻帧平均相似度：", np.mean([sim[i, i + 1] for i in range(len(t) - 1)]).round(3))

> 问题：对角线附近的亮块有多宽？这就是第 3 天 `block_size` 至少要设多大。